# COMPILER DESIGN - UNITS 4, 5, 6
## Complete Question Bank with Detailed Answers & Worked Numericals
### (30 marks total)

---

# UNIT 4: SYNTAX DIRECTED TRANSLATION & INTERMEDIATE CODE GENERATION (10 marks)

## Q1: What is Three-Address Code (TAC)? Explain its forms and importance.

### Answer:

**Three-Address Code (TAC)** is an intermediate representation of a program where each instruction has at most three operands. It serves as a bridge between high-level source code and low-level assembly code.

### Why TAC?
- **Simplicity:** Easy to translate to assembly language
- **Optimization:** Enables machine-independent code optimizations
- **Portability:** Same TAC can target different machine architectures
- **Debugging:** Intermediate form aids in tracking execution

### Forms of TAC Instructions:

#### 1. **Binary Operation**
```
x = y op z
```
where op is +, −, *, /, %, &, |, ^

Example: `t1 = a + b`

#### 2. **Unary Operation**
```
x = op y
```
where op is −, !, ~

Example: `t2 = -x` or `t3 = !flag`

#### 3. **Assignment**
```
x = y
```
Simple copy operation

Example: `result = t1`

#### 4. **Conditional Jump**
```
if x relop y goto L
```
where relop is <, <=, >, >=, ==, !=

Example: `if i < n goto LOOP_START`

#### 5. **Unconditional Jump**
```
goto L
```

Example: `goto EXIT`

#### 6. **Array Access**
```
x = y[i]          (load from array)
```
```
y[i] = x          (store to array)
```

#### 7. **Function Call**
```
param x           (push parameter)
call p, n         (call function p with n parameters)
x = call p        (get return value)
return x          (return value)
```

### Characteristics of TAC:
- **Temporary variables** (t1, t2, t3...) hold intermediate results
- **Explicit control flow** (labels, jumps)
- **Linear sequence** of instructions
- **No nested operations** (one operation per instruction)

---

## Q2: DETAILED NUMERICAL - Generate TAC for Complex Expression

### Problem 1: Expression with Multiple Operators
Generate TAC for the expression: `x = (a + b) * (c - d) / (e + f)`

### Complete Solution:

#### Step 1: Understand Operator Precedence and Associativity
- Precedence: `*, /` have higher precedence than `+, −`
- Associativity: All are left-associative

#### Step 2: Build Expression Tree (respecting precedence)
```
                 =
                / \\
               x   /
              / \\
             *   +
            / \ / \\
           +  - e  f
          / \ / \\
         a b c d
```

#### Step 3: Postorder Traversal and TAC Generation
Postorder traversal means: left subtree → right subtree → node

| Step | Operation | TAC Instruction | Explanation |
|------|-----------|-----------------|-------------|
| 1 | a + b | `t1 = a + b` | Compute first addition |
| 2 | c - d | `t2 = c - d` | Compute subtraction |
| 3 | t1 * t2 | `t3 = t1 * t2` | Multiply results |
| 4 | e + f | `t4 = e + f` | Compute second addition |
| 5 | t3 / t4 | `t5 = t3 / t4` | Divide |
| 6 | x = t5 | `x = t5` | Assign final result |

#### Final TAC:
```
t1 = a + b
t2 = c - d
t3 = t1 * t2
t4 = e + f
t5 = t3 / t4
x = t5
```

#### Key Observations:
- 5 temporary variables (t1-t5) created
- 6 TAC instructions total
- Order respects operator precedence
- Left-to-right evaluation for same precedence

---

### Problem 2: Control Flow - If-Then-Else Statement

Generate TAC for:
```c
if (x > 5)
    y = 10;
else
    y = 20;
```

### Complete Solution:

#### Approach: Backpatching with Labels
We use labels (L1, L2) to control flow:
- **L1:** Label for "then" branch (true condition)
- **L2:** Label for end of if-else

#### TAC Instructions:

| Line | TAC Instruction | Purpose |
|------|-----------------|----------|
| 1 | `if x > 5 goto L1` | Check condition; if true, jump to true branch |
| 2 | `y = 20` | **Else branch:** assign 20 |
| 3 | `goto L2` | Skip then branch and go to end |
| 4 | `L1: y = 10` | **Then branch:** assign 10 |
| 5 | `L2:` | **End:** continue execution |

#### Explanation:
1. **Condition evaluation:** Line 1 tests `x > 5`
   - If **true:** jump to L1 (line 4)
   - If **false:** continue to line 2 (else branch)
2. **Else branch:** Line 2 assigns `y = 20`
3. **Skip then:** Line 3 unconditional jump to L2 (avoid executing then branch)
4. **Then branch:** Line 4 assigns `y = 10`
5. **End:** Line 5 both branches converge here

#### Execution Paths:
**When x > 5 (condition true):**
```
Line 1: if x > 5 goto L1   ✓ True → jump to L1
Line 4: y = 10              Execute
Line 5: L2:                 Continue
```

**When x ≤ 5 (condition false):**
```
Line 1: if x > 5 goto L1   ✗ False → continue
Line 2: y = 20              Execute
Line 3: goto L2             Jump to L2
Line 5: L2:                 Continue (skip line 4)
```

#### Key Concept: Backpatching
- When generating TAC for `if`, we don't know L1 and L2 addresses yet
- We generate labels symbolically
- During second pass, replace labels with actual instruction addresses

---

### Problem 3: Loop with Array Access

Generate TAC for:
```c
while (i < n) {
    sum = sum + a[i];
    i = i + 1;
}
```

### Complete Solution:

#### Approach:
- **L1:** Loop entry (condition check)
- **L2:** Loop exit (after condition false)
- **Array indexing:** `a[i]` requires computing address: `base_address + (i * element_size)`

#### Assuming:
- Array `a` is 4 bytes per element (int)
- `a` is base address in memory

#### TAC Instructions:

| Line | TAC Instruction | Explanation |
|------|-----------------|-------------|
| 1 | `L1: if i >= n goto L2` | Loop condition (negated for false branch) |
| 2 | `t1 = 4 * i` | Compute array offset (4 bytes per int) |
| 3 | `t2 = a + t1` | Address of a[i] |
| 4 | `t3 = *t2` | Load a[i] into t3 |
| 5 | `t4 = sum + t3` | Compute sum + a[i] |
| 6 | `sum = t4` | Store new sum |
| 7 | `t5 = i + 1` | Increment i |
| 8 | `i = t5` | Update i |
| 9 | `goto L1` | Jump back to condition check |
| 10 | `L2:` | Loop exit |

#### Detailed Explanation:

**Line 1: Loop Condition**
- `if i >= n goto L2` means: if false, continue loop body
- We use negated condition for clarity in TAC

**Lines 2-4: Array Access (sum + a[i])**
- Line 2: Compute memory offset from array index
  - If i=0: t1 = 0 (first element)
  - If i=1: t1 = 4 (second element)
  - If i=2: t1 = 8 (third element)
- Line 3: Add base address `a` to offset
  - t2 = a + t1 gives address of a[i]
- Line 4: Dereference pointer
  - t3 = *t2 loads value from memory at address t2

**Lines 5-6: Update sum**
- Line 5: Compute sum + a[i]
- Line 6: Store back to sum

**Lines 7-8: Loop counter increment**
- Line 7: Compute i + 1
- Line 8: Update loop variable

**Line 9: Jump to condition**
- Unconditional jump back to L1 for next iteration

**Line 10: Loop exit**
- Both label for goto and continued execution after loop

#### Execution Example:
Assume: a=[10, 20, 30], n=3, i=0, sum=0

**Iteration 1 (i=0):**
```
Line 1: if 0 >= 3? No → continue
Line 2: t1 = 4 * 0 = 0
Line 3: t2 = a + 0 = address of a[0]
Line 4: t3 = *t2 = 10
Line 5: t4 = 0 + 10 = 10
Line 6: sum = 10
Line 7: t5 = 0 + 1 = 1
Line 8: i = 1
Line 9: goto L1
```

**Iteration 2 (i=1):**
```
Line 1: if 1 >= 3? No → continue
Line 2: t1 = 4 * 1 = 4
Line 3: t2 = a + 4 = address of a[1]
Line 4: t3 = *t2 = 20
Line 5: t4 = 10 + 20 = 30
Line 6: sum = 30
Line 7: t5 = 1 + 1 = 2
Line 8: i = 2
Line 9: goto L1
```

**Iteration 3 (i=2):**
```
Line 1: if 2 >= 3? No → continue
Line 2: t1 = 4 * 2 = 8
Line 3: t2 = a + 8 = address of a[2]
Line 4: t3 = *t2 = 30
Line 5: t4 = 30 + 30 = 60
Line 6: sum = 60
Line 7: t5 = 2 + 1 = 3
Line 8: i = 3
Line 9: goto L1
```

**Iteration 4 (i=3):**
```
Line 1: if 3 >= 3? Yes → goto L2
Line 10: L2: (exit loop)
```

**Final result: sum = 60**

#### Key Points:
- **Array indexing:** Requires multiplication by element size
- **Pointer dereference:** `*t2` to load value from address
- **Loop structure:** L1 for entry, L2 for exit, goto L1 to continue
- **TAC temporaries:** t1-t5 hold intermediate results

---

## Q3: Syntax-Directed Definition (SDD) and Type Checking

### Answer:

**Syntax-Directed Definition (SDD)** is a context-free grammar augmented with semantic rules. Each production has associated actions that compute attributes.

### Key Concepts:

#### 1. **Synthesized Attributes**
- Computed from children nodes in parse tree
- Pass values **UP** the tree
- Example: `val` attribute of expression node = values of children

#### 2. **Inherited Attributes**
- Passed from parent to child
- Pass values **DOWN** the tree
- Example: `type` attribute for declarations

### Example SDD for Arithmetic Expressions:

```
Production              Semantic Rule
─────────────────────────────────────────
E → E + T              E.val = E₁.val + T.val
E → T                  E.val = T.val
T → T * F              T.val = T₁.val * F.val
T → F                  T.val = F.val
F → ( E )              F.val = E.val
F → num                F.val = num.value
```

### SDD Evaluation for Expression "2 + 3 * 4":

#### Parse Tree:
```
          E
         / \\
        E  T
        |  /|\\
        T T * F
        |  |   |
        F  F num(4)
        |  |   
      num num
      (2) (3)
```

#### Evaluation (Postorder):

| Node | Computation | Value |
|------|-------------|-------|
| num(2) | 2 | 2 |
| F → num(2) | F.val = 2 | 2 |
| T → F | T.val = F.val = 2 | 2 |
| E → T | E₁.val = T.val = 2 | 2 |
| num(3) | 3 | 3 |
| F → num(3) | F.val = 3 | 3 |
| T₁ → F | T₁.val = F.val = 3 | 3 |
| num(4) | 4 | 4 |
| F → num(4) | F.val = 4 | 4 |
| T → T₁ * F | T.val = 3 * 4 = 12 | 12 |
| E → E₁ + T | E.val = 2 + 12 = 14 | **14** |

### Type Checking Example SDD:

```
Production                    Semantic Rule
─────────────────────────────────────────────────────
D → T id                       id.type = T.type
T → int                        T.type = "int"
T → real                       T.type = "real"
E → E₁ + E₂                    if E₁.type == E₂.type then
                                   E.type = E₁.type
                               else
                                   E.type = "type error"
E → id                         E.type = id.type
```

**Example:** Check if `int x; real y; z = x + y;` is type-correct
- E₁.type = int (x is int)
- E₂.type = real (y is real)
- E.type = type error (int ≠ real)

---

# UNIT 5: CODE OPTIMIZATION (10 marks)

## Q1: Explain Local and Global Code Optimizations. Give Examples.

### Answer:

**Code Optimization** improves program performance by reducing execution time, memory usage, or power consumption, without changing program semantics.

### Types of Optimizations:

---

## Local Optimizations (Within Basic Block)

A **basic block** is a sequence of consecutive instructions with:
- Single entry point (first instruction)
- Single exit point (last instruction)
- No jumps in middle

### 1. **Constant Folding**
**Definition:** Evaluate constant expressions at compile time instead of runtime.

**Example:**
```
Before:  x = 5 * 3 + 2
After:   x = 17
```

**Benefit:** Eliminates runtime arithmetic, saves instructions

### 2. **Constant Propagation**
**Definition:** Replace variables with known constant values throughout code.

**Example:**
```c
Before:  x = 5;
         y = x + 3;
         z = x * 2;
         
After:   x = 5;
         y = 8;
         z = 10;
```

**Benefit:** Reduces number of variable accesses, enables further optimizations

### 3. **Strength Reduction**
**Definition:** Replace expensive operations with cheaper equivalent operations.

**Examples:**
```
Before:  x = y * 2
After:   x = y << 1     (bit shift is faster)

Before:  x = y * 4
After:   x = y << 2

Before:  x = y / 2
After:   x = y >> 1     (right shift faster than division)
```

**Benefit:** Reduces instruction latency

### 4. **Dead Code Elimination**
**Definition:** Remove unreachable code or unused variable assignments.

**Examples:**
```c
Before:  x = 5;
         y = 10;
         goto L;
         z = 20;         // Dead code (unreachable)
         L: print(y);
         
After:   x = 5;
         y = 10;
         goto L;
         L: print(y);
```

```c
Before:  x = 5;         // Dead (x never used)
         y = 10;
         print(y);
         
After:   y = 10;
         print(y);
```

**Benefit:** Reduces code size, removes unnecessary computations

### 5. **Algebraic Simplification**
**Definition:** Apply algebraic identities to simplify expressions.

**Examples:**
```
x + 0 → x
x * 1 → x
x * 0 → 0
x - x → 0
x & x → x
x | 0 → x
```

---

## Global Optimizations (Across Basic Blocks)

### 1. **Common Subexpression Elimination (CSE)**
**Definition:** Identify identical expressions computed multiple times and reuse result.

**Example:**
```c
Before:  t1 = a + b;
         t2 = c * d;
         t3 = a + b;    // Same as t1
         t4 = t1 + t3;
         
After:   t1 = a + b;
         t2 = c * d;
         t3 = t1;       // Reuse t1
         t4 = t1 + t3;
```

**TAC Representation:**
```
Before:  t1 = a + b
         t2 = c * d
         t3 = a + b
         t4 = t1 + t3
         
After:   t1 = a + b
         t2 = c * d
         t3 = t1
         t4 = t1 + t3  →  t4 = 2 * t1
```

**Benefit:** Reduces computations, saves registers

### 2. **Loop Invariant Code Motion**
**Definition:** Move code that computes same value every iteration outside loop.

**Example:**
```c
Before:  for (i = 0; i < n; i++) {
             x = y * z;      // Computed every iteration!
             a[i] = x + i;
         }
         
After:   x = y * z;             // Compute once
         for (i = 0; i < n; i++) {
             a[i] = x + i;       // Reuse x
         }
```

**Why it works:** y * z doesn't change in loop; same result every iteration

**Benefit:** Huge performance gain; if loop runs 1000 times, saves 999 multiplications

### 3. **Induction Variable Elimination**
**Definition:** Replace expensive operations on loop counters with cheaper ones.

**Example:**
```c
Before:  for (i = 0; i < n; i++) {
             t = i * 4;      // Multiply every iteration
             a[t] = ...
         }
         
After:   t = 0;
         for (i = 0; i < n; i++) {
             a[t] = ...
             t = t + 4;      // Add (cheaper than multiply)
         }
```

**Benefit:** Reduces loop overhead

---

## Q2: DETAILED NUMERICAL - Data-Flow Analysis (Reaching Definitions)

### Theory:

**Reaching Definitions Analysis:** Determines which variable definitions can reach a given program point.

**Definition:** A definition d reaches point p if there is a path from d to p with no redefinition of that variable.

**Use:** Enables constant propagation, dead code elimination, common subexpression elimination

---

### Problem:

Analyze the program below for reaching definitions:

```
B1: x = 5        (definition x₁)
B2: y = 10       (definition y₂)
B3: if (x > 0) goto B4 else goto B5
B4: x = 20       (definition x₄)
B5: (empty, just pass through)
B6: z = x + y    (use x, use y)
```

### Control Flow Graph:

```
      B1(x=5)
        |
      B2(y=10)
        |
      B3(if x>0)
      / \\
    B4     B5
  (x=20) (empty)
    \ /
    B6(z=x+y)
```

---

### Solution:

#### Step 1: Define GEN and KILL sets for each block

**GEN[B]:** Definitions that are **generated (created)** in block B and not killed
**KILL[B]:** Definitions of same variable that are **overwritten** by this block

| Block | Generated Definitions | Killed Definitions |
|-------|----------------------|--------------------|
| B1 | {x₁} | {x₄} (x₄ is killed by x₁) |
| B2 | {y₂} | {} (no other y definitions) |
| B3 | {} | {} (no definitions) |
| B4 | {x₄} | {x₁} (x₁ is killed by x₄) |
| B5 | {} | {} (empty block) |
| B6 | {z₆} | {} (no other z definitions) |

---

#### Step 2: Data-Flow Equations

For each block B:
```
IN[B] = ∪ OUT[P] for all predecessors P of B
OUT[B] = GEN[B] ∪ (IN[B] - KILL[B])
```

**Explanation:**
- **IN[B]:** All definitions reaching start of block (union of all predecessors' outputs)
- **OUT[B]:** All definitions reaching end of block
  - Include generated definitions
  - Include incoming definitions that aren't killed (overwritten)

---

#### Step 3: Iterative Fixed-Point Solution

**Initialize:** All IN and OUT sets to empty

**Iteration 1:**

```
B1:
  IN[B1] = {}              (no predecessors)
  OUT[B1] = {x₁} ∪ ({} - {x₄}) = {x₁}

B2:
  IN[B2] = OUT[B1] = {x₁}
  OUT[B2] = {y₂} ∪ ({x₁} - {}) = {x₁, y₂}

B3:
  IN[B3] = OUT[B2] = {x₁, y₂}
  OUT[B3] = {} ∪ ({x₁, y₂} - {}) = {x₁, y₂}

B4:
  IN[B4] = OUT[B3] = {x₁, y₂}     (from B3→B4)
  OUT[B4] = {x₄} ∪ ({x₁, y₂} - {x₁}) = {x₄, y₂}

B5:
  IN[B5] = OUT[B3] = {x₁, y₂}     (from B3→B5)
  OUT[B5] = {} ∪ ({x₁, y₂} - {}) = {x₁, y₂}

B6:
  IN[B6] = OUT[B4] ∪ OUT[B5] = {x₄, y₂} ∪ {x₁, y₂} = {x₁, x₄, y₂}
           (both x definitions reach B6 because of two paths)
  OUT[B6] = {z₆} ∪ ({x₁, x₄, y₂} - {}) = {x₁, x₄, y₂, z₆}
```

**Iteration 2:** (same results, convergence reached)

---

#### Final Result:

| Block | IN Set | OUT Set |
|-------|--------|----------|
| B1 | {} | {x₁} |
| B2 | {x₁} | {x₁, y₂} |
| B3 | {x₁, y₂} | {x₁, y₂} |
| B4 | {x₁, y₂} | {x₄, y₂} |
| B5 | {x₁, y₂} | {x₁, y₂} |
| B6 | {x₁, x₄, y₂} | {x₁, x₄, y₂, z₆} |

---

#### Interpretation at B6:

Statement: `z = x + y`

**At B6, variable x can be defined at:**
- x₁ (value 5) from B1 if path B1→B2→B3→B5→B6 is taken
- x₄ (value 20) from B4 if path B1→B2→B3→B4→B6 is taken

**At B6, variable y can be defined at:**
- y₂ (value 10) from B2

**Analysis uses:**
- **Constant propagation:** If path determined, replace x with constant
- **Dead code:** x₁ may be dead in path B1→B2→B3→B4→B6 (overwritten before use)
- **CSE:** If `x + y` appears elsewhere with same definitions, reuse result

---

## Q3: NUMERICAL - Live Variable Analysis

### Theory:

**Live Variable Analysis:** A variable is **live** at a point if its value might be used on some path from that point.

**Use:** Register allocation (don't spill live variables), dead code elimination

---

### Problem:

Perform live variable analysis on:

```
B1: a = 5        (a defined)
B2: b = 10       (b defined)
B3: c = a + b    (a, b used; c defined)
B4: d = c * 2    (c used; d defined)
B5: e = d + a    (d, a used; e defined)
B6: print(e)     (e used)
```

---

### Solution:

#### Key Concept:
**Liveness works BACKWARD:** Start from end, propagate backward

For each statement:
```
USE[s] = variables used in statement s
DEF[s] = variables defined in statement s

LIVEOUT[s] = union of LIVEIN of successors
LIVEIN[s] = USE[s] ∪ (LIVEOUT[s] - DEF[s])
```

---

#### Extract USE and DEF sets:

| Statement | USE | DEF |
|-----------|-----|-----|
| a = 5 | {} | {a} |
| b = 10 | {} | {b} |
| c = a + b | {a, b} | {c} |
| d = c * 2 | {c} | {d} |
| e = d + a | {d, a} | {e} |
| print(e) | {e} | {} |

---

#### Backward Analysis:

**At B6 (print(e)):**
```
USE[B6] = {e}
DEF[B6] = {}
LIVEOUT[B6] = {} (end of program)
LIVEIN[B6] = {e} ∪ ({} - {}) = {e}
```

**At B5 (e = d + a):**
```
USE[B5] = {d, a}
DEF[B5] = {e}
LIVEOUT[B5] = LIVEIN[B6] = {e}
LIVEIN[B5] = {d, a} ∪ ({e} - {e}) = {d, a}
```

**At B4 (d = c * 2):**
```
USE[B4] = {c}
DEF[B4] = {d}
LIVEOUT[B4] = LIVEIN[B5] = {d, a}
LIVEIN[B4] = {c} ∪ ({d, a} - {d}) = {c, a}
```

**At B3 (c = a + b):**
```
USE[B3] = {a, b}
DEF[B3] = {c}
LIVEOUT[B3] = LIVEIN[B4] = {c, a}
LIVEIN[B3] = {a, b} ∪ ({c, a} - {c}) = {a, b}
```

**At B2 (b = 10):**
```
USE[B2] = {}
DEF[B2] = {b}
LIVEOUT[B2] = LIVEIN[B3] = {a, b}
LIVEIN[B2] = {} ∪ ({a, b} - {b}) = {a}
```

**At B1 (a = 5):**
```
USE[B1] = {}
DEF[B1] = {a}
LIVEOUT[B1] = LIVEIN[B2] = {a}
LIVEIN[B1] = {} ∪ ({a} - {a}) = {}
```

---

#### Final Live Variable Analysis:

| Statement | LIVEIN | LIVEOUT |
|-----------|--------|----------|
| a = 5 | {} | {a} |
| b = 10 | {a} | {a, b} |
| c = a + b | {a, b} | {a, c} |
| d = c * 2 | {a, c} | {a, d} |
| e = d + a | {a, d} | {e} |
| print(e) | {e} | {} |

---

#### Analysis & Implications:

**Live Ranges:**
- **a:** lives from B1 through B5 (used multiple times)
- **b:** lives from B2 through B3 only (used once in B3)
- **c:** lives from B3 through B4 (short live range)
- **d:** lives from B4 through B5 (short live range)
- **e:** lives from B5 through B6 (shortest, only used in print)

**Register Allocation Implications:**
- **a:** Must keep in register/memory from B1 to B5 (5 instructions)
- **b:** Register can be reused after B3
- **c:** Register can be reused after B4
- **d:** Register can be reused after B5
- **e:** Register only needed for B5-B6

**Dead Code Elimination:**
- No dead code here (all variables are eventually used)
- But if print(e) was removed, B5 becomes dead (e defined but never used)

---

# UNIT 6: CODE GENERATION (10 marks)

## Q1: Explain Main Tasks of Code Generation with Examples

### Answer:

**Code Generation** is the final major phase that converts intermediate code (TAC) to target machine code.

### Main Tasks:

---

## Task 1: Instruction Selection

**Definition:** Choose appropriate target machine instructions for each TAC statement.

**Considerations:**
- Addressing modes (register, immediate, memory, indexed)
- Available operations
- Instruction latency

**Examples:**

**Example 1: Simple Assignment**
```
TAC:      t1 = a + b

Option A (Memory operands):
  LOAD  R1, [a]      ; Load a from memory
  LOAD  R2, [b]      ; Load b from memory
  ADD   R1, R1, R2   ; R1 = R1 + R2
  
Option B (Immediate):
  LOAD  R1, 5        ; If a=5 constant
  LOAD  R2, 3        ; If b=3 constant
  ADD   R1, R1, R2   ; R1 = 8
  
Choice: Depends on whether a, b are constants or variables
```

**Example 2: Array Access**
```
TAC:      t2 = a[i]

Option A (Manual indexing):
  LOAD  R1, i        ; Load index
  LOAD  R2, 4        ; Load element size
  MUL   R1, R1, R2   ; R1 = i * 4
  LOAD  R3, a        ; Load base address
  ADD   R3, R3, R1   ; R3 = a + offset
  LOAD  R2, [R3]     ; Load from memory
  
Option B (Indexed addressing):
  LOAD  R1, i        ; Load index
  LOAD  R2, [a + R1*4]  ; Indexed addressing (if supported)
  
Choice: Option B is better (fewer instructions)
```

---

## Task 2: Register Allocation

**Definition:** Assign variables and temporaries to physical CPU registers.

**Key Decisions:**
1. Which variables stay in registers (frequently used)
2. Which values spill to memory (register pressure)
3. Which registers can be reused (dead values)

**Approaches:**
- **Linear scan:** Simple, fast
- **Graph coloring:** Optimal but complex

**Example:**
```
TAC:
  t1 = a + b
  t2 = c * d
  t3 = t1 - t2
  
Available registers: R1, R2, R3 (3 registers)

Live variable analysis:
  After t1: t1 is live
  After t2: t1, t2 are live (2 live simultaneously)
  After t3: t3 is live
  
Allocation:
  a → R1
  b → R2
  c → R1 (reuse R1; a is dead)
  d → R2 (reuse R2; b is dead)
  t1 → R1 (reuse R1; a is dead)
  t2 → R2 (reuse R2; c, d are dead)
  t3 → R3
```

---

## Task 3: Instruction Scheduling

**Definition:** Reorder instructions to:
- Minimize pipeline hazards and stalls
- Maximize instruction-level parallelism
- Hide memory latency

**Example:**
```
Before (with stalls):
  LOAD   R1, [addr1]   ; Load from memory (latency: 3 cycles)
  ADD    R2, R1, R3    ; Stall! R1 not ready yet
  ADD    R4, R5, R6    ; Could execute in parallel
  
After (scheduled):
  LOAD   R1, [addr1]   ; Cycle 1
  ADD    R4, R5, R6    ; Cycle 1 (no dependency, executes in parallel)
  ADD    R2, R1, R3    ; Cycle 4 (R1 ready by now)
```

**Benefit:** Executes in 3 cycles instead of 5

---

## Task 4: Peephole Optimization

**Definition:** Local post-processing that removes redundant generated instructions.

**Examples:**

**Pattern 1: Redundant Move**
```
Before:  MOV  R1, R2
         MOV  R2, R1
After:   MOV  R1, R2
```

**Pattern 2: Useless Move**
```
Before:  MOV  R1, R1   ; Move to itself!
After:   (eliminate)
```

**Pattern 3: Unreachable Code**
```
Before:  GOTO L
         x = 5          ; Unreachable
         L: ...
After:   GOTO L
         L: ...
```

**Pattern 4: Jump Optimization**
```
Before:  JMP L1
         L1: JMP L2
         L2: ...
After:   JMP L2
         L2: ...
```

---

## Q2: DETAILED NUMERICAL - TAC to Assembly Code Generation

### Problem:

Convert the TAC below to assembly code with complete register allocation:

```
Given TAC:
  t1 = a + b
  t2 = c - d
  t3 = t1 * t2
  t4 = t3 / e
  result = t4
```

**Target Machine:**
- Registers: R1, R2, R3 (3 general-purpose registers)
- All variables are in memory with addresses [a], [b], [c], [d], [e], [result]
- Operations: LOAD, STORE, ADD, SUB, MUL, DIV

---

### Solution:

#### Step 1: Live Variable Analysis

Determine which variables are live (will be used later) at each point:

| After Statement | LIVEOUT |
|------------|----------|
| t1 = a + b | {t1, t2, t3, t4} (all used later) |
| t2 = c - d | {t1, t2, t3, t4} |
| t3 = t1 * t2 | {t3, t4} (t1, t2 no longer used) |
| t4 = t3 / e | {t4} (t3 no longer used) |
| result = t4 | {} (nothing live after) |

**Key Observation:** Maximum 2 values live simultaneously (t1 & t2 after line 1)

---

#### Step 2: Register Allocation

Assign values to registers based on live ranges:

```
Value   Live Range        Assigned Register   Reason
─────────────────────────────────────────────────────────
a       Line 1             R1 (temporary)      Load for computation
b       Line 1             R2 (temporary)      Load for computation
t1      Lines 1-3          R1                  Can reuse R1 (a dead)
c       Line 2             R1                  Can reuse R1 (t1 already computed)
d       Line 2             R2                  Can reuse R2 (b dead)
t2      Lines 2-3          R2                  Computed in R2
t3      Lines 3-4          R3                  New register
e       Line 4             R1                  Temporary
t4      Lines 4-5          R1                  Result of division
result  Line 5             Memory              Final store
```

---

#### Step 3: Assembly Code Generation

| TAC Instruction | Assembly Code | Register State | Explanation |
|-----------------|---------------|---------------|-----------|
| t1 = a + b | LOAD R1, [a] | R1=a | Load a |
| | LOAD R2, [b] | R1=a, R2=b | Load b |
| | ADD R1, R1, R2 | R1=t1 | Compute t1 in R1 |
| t2 = c - d | LOAD R1, [c] | R1=c (t1 pushed/saved) | Load c (R1 reused) |
| | LOAD R2, [d] | R1=c, R2=d | Load d |
| | SUB R2, R1, R2 | R1=?, R2=t2 | Compute t2 in R2 |
| t3 = t1 * t2 | ... (t1 must be restored from stack) | | |
| | MUL R3, R1, R2 | R3=t3 | Compute t3 |
| t4 = t3 / e | LOAD R1, [e] | R1=e | Load e |
| | DIV R1, R3, R1 | R1=t4 | Compute division |
| result = t4 | STORE [result], R1 | | Store result |

**Problem:** Register pressure — we have 2 values live (t1, t2) but only 3 registers. When computing t3 = t1 * t2, we need t1 and t2, but we've overwritten R1 with c!

---

#### Step 3 (Revised): With Spilling

Since we can't keep all values in registers, we use **stack spilling**:

```
Stack Frame:
  SP - 4: [spill_t1]      (temporary storage for t1)
  SP - 8: [spill_t2]      (temporary storage for t2)
```

#### Complete Assembly Code:

```assembly
; Instruction 1-3: t1 = a + b
LOAD    R1, [a]         ; R1 = a
LOAD    R2, [b]         ; R2 = b
ADD     R1, R1, R2      ; R1 = t1 = a + b
STORE   [SP-4], R1      ; Spill t1 to stack

; Instruction 4-7: t2 = c - d
LOAD    R1, [c]         ; R1 = c
LOAD    R2, [d]         ; R2 = d
SUB     R2, R1, R2      ; R2 = t2 = c - d
STORE   [SP-8], R2      ; Spill t2 to stack

; Instruction 8-11: t3 = t1 * t2
LOAD    R1, [SP-4]      ; R1 = t1 (restore from stack)
LOAD    R2, [SP-8]      ; R2 = t2 (restore from stack)
MUL     R3, R1, R2      ; R3 = t3 = t1 * t2

; Instruction 12-14: t4 = t3 / e
LOAD    R1, [e]         ; R1 = e
DIV     R1, R3, R1      ; R1 = t4 = t3 / e

; Instruction 15: result = t4
STORE   [result], R1    ; Store result
```

---

#### Step 4: Analysis

**Instruction Count:** 15 instructions total

**Register Usage:**
- R1: Heavy use (6 LOAD/STORE operations)
- R2: Moderate use (4 operations)
- R3: Light use (1 MUL)

**Spills:** 2 spills (t1, t2) due to register pressure

**Memory Traffic:** 9 LOAD/STORE operations (could be optimized further)

**Optimization Opportunity:** 
- With 4+ registers available, could eliminate spilling
- Would reduce from 15 to 11 instructions

---

## Q3: Activation Records and Function Calls (Detailed)

### Theory:

**Activation Record (Stack Frame):** Data structure on the call stack storing all information needed to execute a function.

---

### Typical Activation Record Layout (Intel x86 calling convention):

```
Stack grows downward ↓

┌───────────────────────────┐
│ Return Address (from CALL)│  FP + 4
├───────────────────────────┤
│ Saved Frame Pointer (EBP) │  FP (EBP points here)
├───────────────────────────┤
│ Saved Registers           │  FP - 4, FP - 8, ...
├───────────────────────────┤
│ Local Variables           │  FP - 12, FP - 16, ...
├───────────────────────────┤
│ Temporary Storage         │
├───────────────────────────┤
│ Outgoing Arguments        │  (for next function call)
└───────────────────────────┘ ← SP (current stack pointer)
```

### Standard Calling Convention:

| Aspect | Convention |
|--------|------------|
| Parameter passing | Push on stack (right-to-left) |
| Return address | Pushed by CALL instruction |
| Caller-saved | EAX, ECX, EDX (caller must save) |
| Callee-saved | EBX, ESI, EDI, EBP (callee must save/restore) |
| Return value | In EAX |
| Stack cleanup | Caller cleans stack after return |

---

## Q4: DETAILED NUMERICAL - Function Prologue, Body, and Epilogue

### Problem:

Write complete assembly code for the function:

```c
int compute(int x, int y) {
    int temp = x + y;
    int result = temp * 2;
    return result;
}
```

### Solution:

#### Step 1: Memory Layout

```
Stack Frame (in bytes, assuming 32-bit integers):

FP + 12:  [Parameter y]      (4 bytes)
FP + 8:   [Parameter x]      (4 bytes)
FP + 4:   [Return Address]   (4 bytes, pushed by CALL)
FP:       [Saved EBP]        (4 bytes)
FP - 4:   [temp]             (4 bytes)
FP - 8:   [result]           (4 bytes)
SP → (FP - 8)
```

---

#### Step 2: Prologue (Setup stack frame)

```assembly
compute:
    PUSH    EBP             ; Save caller's frame pointer (1)
    MOV     EBP, ESP        ; Set up new frame pointer (2)
    SUB     ESP, 8          ; Allocate 8 bytes for locals (temp, result) (3)
```

**Explanation:**
- Line 1: PUSH EBP
  - Saves caller's frame pointer
  - Stack pointer decreases by 4
  - ESP now points to saved EBP
  
- Line 2: MOV EBP, ESP
  - EBP now points to saved EBP location
  - This becomes reference point for accessing parameters and locals
  
- Line 3: SUB ESP, 8
  - Allocates 8 bytes on stack (2 local variables)
  - ESP moves down by 8
  - Now points below all local variables

**Stack after prologue:**
```
  EBP → [saved caller's EBP]
  [return address]
  [parameter y]
  [parameter x]
  [temp storage]
  [result storage]
  ESP → (empty space for future use)
```

---

#### Step 3: Function Body

```assembly
    ; temp = x + y
    MOV     EAX, [EBP+8]    ; EAX = x
    MOV     ECX, [EBP+12]   ; ECX = y
    ADD     EAX, ECX        ; EAX = x + y
    MOV     [EBP-4], EAX    ; temp = EAX
    
    ; result = temp * 2
    MOV     EAX, [EBP-4]    ; EAX = temp
    IMUL    EAX, 2          ; EAX = temp * 2 (signed multiply)
    MOV     [EBP-8], EAX    ; result = EAX
    
    ; Prepare return value
    MOV     EAX, [EBP-8]    ; EAX = result (return in EAX)
```

**Step-by-step explanation:**

**Getting parameters:**
```
MOV EAX, [EBP+8]    ; x is at EBP+8 (parameters above saved EBP)
MOV ECX, [EBP+12]   ; y is at EBP+12 (y is further down stack)
```

**Computing x + y:**
```
ADD EAX, ECX        ; EAX = EAX + ECX = x + y
```

**Storing temp:**
```
MOV [EBP-4], EAX   ; temp is at FP-4 (locals are below saved EBP)
```

**Computing temp * 2:**
```
MOV EAX, [EBP-4]   ; Load temp back
IMUL EAX, 2         ; Multiply by 2 (IMUL for signed)
```

**Storing result:**
```
MOV [EBP-8], EAX   ; result is at FP-8
```

**Prepare return value:**
```
MOV EAX, [EBP-8]   ; Put result in EAX (calling convention)
```

---

#### Step 4: Epilogue (Restore stack frame)

```assembly
    MOV     ESP, EBP        ; Deallocate local variables
    POP     EBP             ; Restore caller's frame pointer
    RET                     ; Pop return address and jump
```

**Explanation:**

**Line 1: MOV ESP, EBP**
- Moves ESP back to where EBP is
- Effectively deallocates all local variables
- Now ESP and EBP point to same location (saved EBP)

**Line 2: POP EBP**
- Pops saved EBP from stack into EBP register
- Restores caller's frame pointer
- ESP automatically increments by 4
- Now ESP points to return address

**Line 3: RET**
- Pops return address from stack
- Jumps to return address
- Stack restored to state before CALL instruction

---

#### Complete Function:

```assembly
compute:
    ; Prologue
    PUSH    EBP             ; Save caller's EBP
    MOV     EBP, ESP        ; Set up new EBP
    SUB     ESP, 8          ; Allocate 8 bytes for locals
    
    ; Function body
    MOV     EAX, [EBP+8]    ; EAX = x
    MOV     ECX, [EBP+12]   ; ECX = y
    ADD     EAX, ECX        ; EAX = x + y
    MOV     [EBP-4], EAX    ; temp = EAX
    
    MOV     EAX, [EBP-4]    ; EAX = temp
    IMUL    EAX, 2          ; EAX = temp * 2
    MOV     [EBP-8], EAX    ; result = EAX
    
    ; Prepare return value
    MOV     EAX, [EBP-8]    ; EAX = result
    
    ; Epilogue
    MOV     ESP, EBP        ; Deallocate locals
    POP     EBP             ; Restore caller's EBP
    RET                     ; Return to caller
```

---

#### Example Execution:

**Caller code:**
```assembly
PUSH    5               ; Push y = 5
PUSH    3               ; Push x = 3
CALL    compute         ; Call function (pushes return address)
```

**Stack after CALL:**
```
Before CALL:  ESP → [param y: 5]
              [param x: 3]
              
After CALL:   ESP → [return addr]
              [param y: 5]
              [param x: 3]
```

**In prologue:**
```
After PUSH EBP: ESP → [saved caller's EBP]
                [return addr]
                [param y: 5]
                [param x: 3]
                
After MOV EBP, ESP: EBP → [saved caller's EBP]
                    
After SUB ESP, 8: ESP → [free space for locals]
                  EBP → [saved caller's EBP]
                  [return addr]
                  [param y: 5]        (EBP+12)
                  [param x: 3]        (EBP+8)
                  [temp]              (EBP-4)
                  [result]            (EBP-8)
```

**During execution (assuming x=3, y=5):**
```
EAX = [EBP+8] = 3
ECX = [EBP+12] = 5
EAX = 3 + 5 = 8
[EBP-4] = 8              (temp = 8)
EAX = [EBP-4] = 8
EAX = 8 * 2 = 16
[EBP-8] = 16             (result = 16)
EAX = [EBP-8] = 16      (return value)
```

**Epilogue:**
```
ESP = EBP               (deallocate locals)
POP EBP                 (restore caller's EBP)
RET                     (jump to caller, ESP += 4)

Back in caller:
  Stack cleanup (if needed):
  ADD ESP, 8              (remove parameters: 4 + 4 bytes)
  
  EAX contains 16 (result)
```

**Final result:** compute(3, 5) returns 16
- temp = 3 + 5 = 8
- result = 8 * 2 = 16

---

# EXAM STRATEGY & FINAL TIPS

## Time Management (Units 4-6: 30 marks, ~75 minutes)

| Unit | Time | Marks | Strategy |
|------|------|-------|----------|
| Unit 4 | 20 min | 10 | TAC generation + SDD |
| Unit 5 | 25 min | 10 | Optimizations + data-flow analysis |
| Unit 6 | 30 min | 10 | Code generation + register allocation + activation records |

---

## Question Types Expected:

### Unit 4 (Syntax Directed Translation & Intermediate Code)
✓ Define TAC and its forms  
✓ Generate TAC for expressions (with operator precedence)  
✓ Generate TAC for control structures (if-else, loops)  
✓ TAC with array access and pointers  
✓ Syntax-directed definitions and type checking  

### Unit 5 (Code Optimization)
✓ Explain optimization techniques (folding, CSE, loop invariant)  
✓ Data-flow analysis (reaching definitions, live variables)  
✓ Basic block CFG construction  
✓ Iterative data-flow solution  
✓ Use results for optimization decisions  

### Unit 6 (Code Generation)
✓ Instruction selection and addressing modes  
✓ Register allocation (linear scan, graph coloring)  
✓ Live variable analysis for register decisions  
✓ Spilling when registers insufficient  
✓ Activation records and calling conventions  
✓ Function prologue/epilogue code  

---

## How to Solve Numericals:

### TAC Generation Questions:
1. **Identify operators** and their precedence
2. **Draw parse tree** respecting precedence
3. **Postorder traversal** to generate TAC
4. **Label each temporary** (t1, t2, ...)
5. **Show all steps clearly**

### Data-Flow Analysis Questions:
1. **Identify GEN and KILL** sets for each block
2. **Write data-flow equations** clearly
3. **Iterate to convergence** (usually 2-3 iterations)
4. **Show IN/OUT sets** for each block
5. **Interpret results** (what does it mean?)

### Code Generation Questions:
1. **Perform live variable analysis** first
2. **Decide register allocation** based on live ranges
3. **Mark spills if needed**
4. **Generate assembly step-by-step**
5. **Label registers clearly**

### Activation Record Questions:
1. **Draw stack frame layout** to scale
2. **Show each field** (return addr, saved regs, locals, params)
3. **Write prologue, body, epilogue** separately
4. **Explain each instruction**
5. **Show stack evolution** before/after CALL, at entry, at exit

---

## Common Exam Mistakes to Avoid:

❌ **TAC:** Forgetting to use temporaries, nesting operations  
❌ **TAC:** Wrong operator precedence in expressions  
❌ **Data-flow:** Forgetting to iterate to convergence  
❌ **Data-flow:** Confusing IN/OUT or gen/kill  
❌ **Register:** Allocating more registers than available  
❌ **Register:** Not accounting for live ranges  
❌ **Activation:** Wrong stack offsets (FP+8 vs FP-4)  
❌ **Activation:** Forgetting prologue or epilogue  

---

## Key Formulas & Rules:

### TAC:
- Each statement: at most 3 operands
- Operator precedence: `*, /` > `+, −`
- Left-to-right for same precedence

### Data-Flow:
```
OUT[B] = GEN[B] ∪ (IN[B] - KILL[B])
IN[B] = ∪ OUT[P] for predecessors P
```

### Register Allocation:
- Max simultaneous live variables determines minimum registers needed
- Reuse registers when variables are dead
- Spill to memory if insufficient registers

### Activation Records:
- Parameters at FP + positive offset
- Locals at FP - negative offset
- Return address and saved FP below parameters

---

## Final Checklist Before Exam:

☐ TAC generation for all expression types (arithmetic, logical, etc.)  
☐ TAC for control flow (if, if-else, while, for)  
☐ Array indexing in TAC  
☐ Data-flow analysis (reaching defs, live vars)  
☐ Basic block identification and CFG  
☐ Register allocation strategies  
☐ Instruction selection examples  
☐ Activation record layout  
☐ Prologue/epilogue generation  
☐ Calling conventions (parameter passing, return values)  

---

**End of Units 4, 5, 6 Complete Question Bank**